# Phase 1 QLoRA Training

This notebook runs the first QLoRA baseline for the phase-1 structured-output task.

Recommended default:

- model: `Qwen/Qwen2.5-3B-Instruct`
- train file: `data/processed/phase1_sft_train.jsonl`
- val file: `data/processed/phase1_sft_val.jsonl`

In [8]:
# Run this once in a fresh Jupyter environment if dependencies are missing.
# Comment it out after the environment is stable.

# %pip install -q transformers datasets peft accelerate trl bitsandbytes pyyaml jsonschema

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import os


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root from current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

PROJECT_ROOT = /home/lyan11/small-llm-structured-posttraining


In [2]:
import torch
import platform
import sys

print('python =', sys.version)
print('platform =', platform.platform())
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    print('gpu_count =', torch.cuda.device_count())
    print('bf16_supported =', torch.cuda.is_bf16_supported())

python = 3.13.11 | packaged by conda-forge | (main, Dec  6 2025, 11:24:03) [GCC 14.3.0]
platform = Linux-6.8.0-106-generic-x86_64-with-glibc2.39
cuda_available = True
gpu = NVIDIA H200 NVL
gpu_count = 1
bf16_supported = True


In [3]:
import yaml
from pathlib import Path

CONFIG_PATH = Path('configs/train/lora_phase1_qwen3b.yaml')
# Fallback if needed:
# CONFIG_PATH = Path('configs/train/lora_phase1_smollm2_1p7b.yaml')

with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

config

{'experiment_name': 'qwen25_3b_phase1_qlora_v1',
 'task_name': 'ticket_structured_output',
 'model': {'base_model': 'Qwen/Qwen2.5-3B-Instruct', 'max_seq_length': 2048},
 'training': {'method': 'qlora',
  'learning_rate': '2e-4',
  'num_train_epochs': 3,
  'per_device_train_batch_size': 2,
  'gradient_accumulation_steps': 8,
  'warmup_ratio': 0.03},
 'lora': {'r': 16, 'alpha': 32, 'dropout': 0.05},
 'data': {'train_file': 'data/processed/phase1_sft_train.jsonl',
  'val_file': 'data/processed/phase1_sft_val.jsonl'},
 'output': {'save_dir': 'results/checkpoints/qwen25_3b_phase1_qlora_v1'}}

In [4]:
from pathlib import Path
import json

train_file = Path(config['data']['train_file'])
val_file = Path(config['data']['val_file'])

print('train exists =', train_file.exists(), train_file)
print('val exists =', val_file.exists(), val_file)

with train_file.open('r', encoding='utf-8') as handle:
    first_record = json.loads(next(handle))

first_record

train exists = True data/processed/phase1_sft_train.jsonl
val exists = True data/processed/phase1_sft_val.jsonl


{'sample_id': 'console_ai_it_helpdesk_tickets-000068-g5ox12x1v',
 'messages': [{'role': 'system',
   'content': 'You are an information extraction model. Return only JSON that matches the requested schema. Do not add explanations or markdown.'},
  {'role': 'user',
   'content': 'Task: extract a structured record for ticket_structured_output.\nSchema name: ticket_schema_v1\nReturn a JSON object only.\nInput text:\nSubject: jane.doe@acme.co VPN Connection Instructions Request\nDescription: I need assistance with connecting to the company VPN for remote work access. Please provide detailed instructions on how to set up and connect to the VPN. Thank you!'},
  {'role': 'assistant',
   'content': '{"ticket_id": "g5ox12x1v", "summary": "jane.doe@acme.co VPN Connection Instructions Request", "category": "question", "priority": "medium", "requires_followup": true, "reporter": {"name": "Jane Doe", "team": null}, "affected_systems": [{"name": "RemoteWork", "component": "remotework"}], "actions_re

In [5]:
from datasets import load_dataset

dataset = load_dataset(
    'json',
    data_files={
        'train': str(train_file),
        'validation': str(val_file),
    },
)

dataset

DatasetDict({
    train: Dataset({
        features: ['sample_id', 'messages', 'metadata'],
        num_rows: 1993
    })
    validation: Dataset({
        features: ['sample_id', 'messages', 'metadata'],
        num_rows: 247
    })
})

In [6]:
from transformers import AutoTokenizer

base_model = config['model']['base_model']
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = 'right'
print('pad_token =', tokenizer.pad_token)
print('eos_token =', tokenizer.eos_token)

pad_token = <|endoftext|>
eos_token = <|im_end|>


In [7]:
def format_chat_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    example["text"] = text
    return example

dataset = dataset.map(format_chat_example)


In [8]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

model.config.use_cache = False
print('model loaded')

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

model loaded


In [9]:
from peft import LoraConfig, TaskType

peft_config = LoraConfig(
    r=config['lora']['r'],
    lora_alpha=config['lora']['alpha'],
    lora_dropout=config['lora']['dropout'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules='all-linear',
)

peft_config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules='all-linear', exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)

In [10]:
from transformers import TrainingArguments
from trl import SFTTrainer

output_dir = config['output']['save_dir']

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=float(config['training']['learning_rate']),
    num_train_epochs=int(config['training']['num_train_epochs']),
    per_device_train_batch_size=int(config['training']['per_device_train_batch_size']),
    per_device_eval_batch_size=int(config['training']['per_device_train_batch_size']),
    gradient_accumulation_steps=int(config['training']['gradient_accumulation_steps']),
    warmup_steps=50,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to='none',
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer,
    peft_config=peft_config,
)


## Pilot Run Suggestion

Before a full run, you can reduce epochs or stop early after confirming the trainer starts correctly.

In [1]:
!nvidia-smi

Thu Mar 19 09:17:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200 NVL                On  |   00000000:00:05.0 Off |                    0 |
| N/A   25C    P0             71W /  600W |       0MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [11]:
train_result = trainer.train()
train_result

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.827603,0.764860
100,0.595239,0.598102
150,0.531135,0.521454
200,0.483581,0.481874
250,0.458381,0.459122
300,0.413020,0.447551
350,0.419578,0.439583


TrainOutput(global_step=375, training_loss=0.6927261422475179, metrics={'train_runtime': 420.624, 'train_samples_per_second': 14.215, 'train_steps_per_second': 0.892, 'total_flos': 2.835424482656256e+16, 'train_loss': 0.6927261422475179})

In [12]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print('saved to', output_dir)

saved to results/checkpoints/qwen25_3b_phase1_qlora_v1


## Optional Quick Generation Check

In [13]:
sample_messages = dataset['validation'][0]['messages']
prompt_text = tokenizer.apply_chat_template(
    sample_messages[:-1],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
generated = model.generate(**inputs, max_new_tokens=256)
decoded = tokenizer.decode(generated[0], skip_special_tokens=True)
print(decoded)

system
You are an information extraction model. Return only JSON that matches the requested schema. Do not add explanations or markdown.
user
Task: extract a structured record for ticket_structured_output.
Schema name: ticket_schema_v1
Return a JSON object only.
Input text:
Subject: Intermittent API Authentication Failures
Description: Hi there, I'm experiencing persistent issues with our enterprise cloud-based application where specific API endpoints intermittently fail to authenticate. This requires a detailed investigation and manual adjustment of the API authentication settings. Could you also inspect the token management protocols? Your assistance would be greatly appreciated! Thank you!! Alex Johnson | Cloud Solutions Specialist Acme Co. ● Remote ● He/Him/His
assistant
{"ticket_id": "80294q57s", "summary": "Intermittent API Authentication Failures", "category": "bug", "priority": "high", "requires_followup": true, "reporter": {"name": "Alex Johnson", "team": null}, "affected_syst

## Test Inference Export

Run the following cells after training to generate predictions for `phase1_test.jsonl` and save them to `results/predictions/`.

In [ ]:
import json
from pathlib import Path

test_path = PROJECT_ROOT / 'data' / 'samples' / 'phase1_test.jsonl'
prediction_dir = PROJECT_ROOT / 'results' / 'predictions'
prediction_dir.mkdir(parents=True, exist_ok=True)
prediction_path = prediction_dir / 'qwen25_3b_phase1_qlora_v1_test.jsonl'

print('test_path =', test_path)
print('prediction_path =', prediction_path)
print('test_exists =', test_path.exists())

In [ ]:
test_records = []
with test_path.open('r', encoding='utf-8') as handle:
    for line in handle:
        test_records.append(json.loads(line))

len(test_records), test_records[0]

In [ ]:
generation_kwargs = {
    'max_new_tokens': 256,
    'do_sample': False,
    'temperature': 1.0,
    'top_p': 1.0,
}

generation_kwargs

In [ ]:
def build_inference_messages(record):
    return [
        {
            'role': 'system',
            'content': 'You are an information extraction model. Return only JSON that matches the requested schema. Do not add explanations or markdown.',
        },
        {
            'role': 'user',
            'content': (
                f"Task: extract a structured record for {record['task_name']}.\n"
                f"Schema name: {record['schema_name']}\n"
                'Return a JSON object only.\n'
                'Input text:\n'
                f"{record['input_text']}"
            ),
        },
    ]


def generate_prediction_text(record):
    messages = build_inference_messages(record)
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, **generation_kwargs)
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


sample_pred = generate_prediction_text(test_records[0])
print(sample_pred)

In [ ]:
predictions = []

for idx, record in enumerate(test_records):
    prediction_text = generate_prediction_text(record)
    prediction_json = None
    try:
        prediction_json = json.loads(prediction_text)
    except json.JSONDecodeError:
        prediction_json = None

    predictions.append(
        {
            'sample_id': record['sample_id'],
            'prediction_text': prediction_text,
            'prediction_json': prediction_json,
            'metadata': {
                'model_name': config['model']['base_model'],
                'experiment_id': config['experiment_name'],
            },
        }
    )

    if (idx + 1) % 25 == 0:
        print(f'generated {idx + 1} / {len(test_records)}')

len(predictions)

In [ ]:
with prediction_path.open('w', encoding='utf-8') as handle:
    for record in predictions:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')

print('saved predictions to', prediction_path)
print('exists =', prediction_path.exists())
print('size =', prediction_path.stat().st_size if prediction_path.exists() else None)

In [ ]:
with prediction_path.open('r', encoding='utf-8') as handle:
    for i, line in enumerate(handle):
        if i >= 2:
            break
        print(json.loads(line))
        print('---')